# Information Theory Basics
**Topic:** Probability & Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go

import ipywidgets as widgets
from ipywidgets import FloatSlider, Dropdown, Output, VBox

from IPython.display import display, clear_output
from scipy import stats

from tkh_utils import PALETTE, FONT, base_layout
from tkh_utils import make_output, make_slider, make_dropdown, display_widget, register_observer

---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** entropy as a measure of uncertainty or surprise in a probability distribution
- **Explain** information gain as the reduction in entropy achieved by a split
- **Interpret** how decision trees use entropy to choose which feature to split on

> **Tip:** Start by moving the probability slider from 0.5 toward 0.0 or 1.0. Watch entropy drop toward zero — that represents a situation with no uncertainty, no information to gain.

---
## How we got here

In *01–03* we learned to calculate the probability of events. Probability tells us how likely something is, but it does not directly answer: how much would learning this outcome surprise us? That is the question information theory answers. A coin flip with 50/50 odds carries maximum uncertainty. A coin that lands heads 99% of the time carries almost no uncertainty, and learning its result is nearly unsurprising.

---
## Why this matters for data science

Information theory shows up in three places you will encounter constantly:

1. **Decision trees** — every split is chosen to maximize information gain (reduction in entropy)
2. **Cross-entropy loss** — the loss function used to train classification models measures how much the model's predicted probabilities diverge from the true labels
3. **Feature selection** — mutual information ranks features by how much knowing the feature reduces uncertainty about the target

---
## Try it yourself

In [2]:
out = make_output()

p_slider = make_slider(0.01, 0.99, 0.01, 0.50, "P(class = 1):")
n_class_dropdown = make_dropdown(
    options=[("Binary (2 classes)", 2), ("3 classes", 3), ("4 classes", 4)],
    value=2,
    description="Classes:"
)

p_full = np.linspace(0.001, 0.999, 400)


def entropy_binary(p):
    q = 1.0 - p
    return -p * np.log2(p) - q * np.log2(q)


def entropy_uniform_k(k):
    """Max entropy for k equal classes."""
    return np.log2(k)


def render(change=None):
    p = p_slider.value
    n_classes = n_class_dropdown.value

    with out:
        clear_output(wait=True)

        if n_classes == 2:
            probs = [p, 1 - p]
        elif n_classes == 3:
            remainder = (1 - p) / 2
            probs = [p, remainder, remainder]
        else:
            remainder = (1 - p) / 3
            probs = [p, remainder, remainder, remainder]

        probs = np.array(probs)
        probs = np.clip(probs, 1e-9, 1)
        h_current = -np.sum(probs * np.log2(probs))
        h_max = np.log2(n_classes)

        h_curve = entropy_binary(p_full)

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=p_full, y=h_curve,
            mode="lines",
            line=dict(color=PALETTE["primary"], width=3),
            name="Binary entropy H(p)",
        ))
        fig.add_trace(go.Scatter(
            x=[p], y=[h_current],
            mode="markers",
            marker=dict(color=PALETTE["secondary"], size=14, symbol="circle"),
            name=f"Current: H = {h_current:.3f} bits",
        ))
        fig.add_hline(
            y=h_max,
            line_dash="dot", line_color=PALETTE["muted"],
            annotation_text=f"Max entropy for {n_classes} classes = {h_max:.2f} bits",
            annotation_position="right",
        )

        layout = base_layout(
            title=f"Entropy H = {h_current:.3f} bits  |  class probs = {[f'{v:.2f}' for v in probs]}",
            xaxis_title="P(class = 1)",
            yaxis_title="Entropy (bits)",
        )
        layout.update(yaxis=dict(range=[-0.05, 2.2]), showlegend=True)
        fig.update_layout(layout)
        display(go.FigureWidget(fig))


register_observer([p_slider, n_class_dropdown], render)

display_widget(VBox([p_slider, n_class_dropdown]), out)
render()

---
## What's happening?

**Entropy** measures how much uncertainty is in a probability distribution. A single number summarizes how mixed up a set of labels is:

$$H(p) = -\sum_{i} p_i \log_2 p_i$$

The units are bits. Key intuitions:

| Distribution | Entropy | Meaning |
|-------------|---------|--------|
| p = [0.5, 0.5] | 1.0 bits | Maximum uncertainty, like a fair coin |
| p = [0.9, 0.1] | 0.47 bits | Mostly predictable |
| p = [1.0, 0.0] | 0.0 bits | Perfectly certain, no information gained by observing |
| p = [0.25, 0.25, 0.25, 0.25] | 2.0 bits | Maximum uncertainty for 4 classes |

### Information gain

When a decision tree evaluates a potential split, it computes how much entropy the split removes:

$$\text{Information Gain} = H(\text{parent}) - \sum_{\text{children}} \frac{n_{\text{child}}}{n} \cdot H(\text{child})$$

A split that separates classes cleanly creates low-entropy children, giving high information gain. The algorithm picks the feature and threshold that maximize this gain.

### Cross-entropy loss

When a classifier outputs predicted probabilities $\hat{p}$ and the true label is $y$, the cross-entropy loss is:

$$\mathcal{L} = -\sum_i y_i \log \hat{p}_i$$

This measures how surprised the model is by the true label. A model that assigns high probability to the correct class is not surprised — and has low loss.

---
## Real-world example: Decision tree splits on customer churn

A telecom company wants to predict customer churn. A decision tree evaluates two candidate splits:

- **Split A** — whether the customer called support more than 3 times
- **Split B** — whether the customer's contract is month-to-month

The chart below computes entropy before and after each split, then shows information gain. The split with the higher gain is what the tree chooses.

Notice:

- The parent node has moderate entropy because churn is not perfectly balanced
- Split B creates one very pure child (long-contract customers rarely churn)
- Split A also helps but creates less pure children, so it has lower information gain

> **Discussion question:** If you added a third split candidate — whether the customer is located in a rural area — but rural vs urban showed no relationship with churn, what would its information gain be? What does that tell you about using mutual information for feature selection?

In [3]:
def entropy(p_pos, n_total):
    if n_total == 0:
        return 0.0
    p = p_pos / n_total
    if p <= 0 or p >= 1:
        return 0.0
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)


def information_gain(parent_pos, parent_n,
                     left_pos, left_n,
                     right_pos, right_n):
    h_parent = entropy(parent_pos, parent_n)
    h_left   = entropy(left_pos, left_n)
    h_right  = entropy(right_pos, right_n)
    weighted = (left_n / parent_n) * h_left + (right_n / parent_n) * h_right
    return h_parent, weighted, h_parent - weighted


# Dataset: 400 customers, 120 churned (30%)
N, CHURNED = 400, 120

# Split A: called support > 3 times
# Left (yes): 80 customers, 56 churned   |  Right (no): 320, 64 churned
h_par_A, h_weighted_A, ig_A = information_gain(CHURNED, N, 56, 80, 64, 320)

# Split B: month-to-month contract
# Left (yes): 180 customers, 102 churned  |  Right (no): 220, 18 churned
h_par_B, h_weighted_B, ig_B = information_gain(CHURNED, N, 102, 180, 18, 220)

splits = ["Parent node", "Split A\n(called >3x)", "Split B\n(month-to-month)"]
colors = [PALETTE["muted"], PALETTE["secondary"], PALETTE["accent"]]
entropies = [h_par_A, h_weighted_A, h_weighted_B]
ig_labels = ["",
             f"IG = {ig_A:.3f} bits",
             f"IG = {ig_B:.3f} bits  ← tree chooses this"]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=splits,
    y=entropies,
    marker_color=colors,
    text=[f"{v:.3f} bits" for v in entropies],
    textposition="outside",
    name="Entropy",
))

layout = base_layout(
    title="Information Gain Comparison — Which Split Does the Tree Choose?",
    xaxis_title="",
    yaxis_title="Entropy (bits)",
)
layout.update(
    yaxis=dict(range=[0, 1.2]),
    showlegend=False,
    annotations=[
        dict(
            x=i, y=entropies[i] + 0.08,
            text=ig_labels[i],
            showarrow=False,
            font=dict(family=FONT["family"], size=11, color="#343A40"),
        )
        for i in range(len(splits))
    ],
)
fig.update_layout(layout)
fig.show()

### Information theory concepts in machine learning

| Concept | Formula | Where it appears |
|---------|---------|------------------|
| Entropy | $H = -\sum p_i \log_2 p_i$ | Decision tree splitting criterion |
| Information gain | $H(\text{parent}) - \sum w_i H(\text{child}_i)$ | CART, ID3, C4.5 tree algorithms |
| Cross-entropy loss | $-\sum y_i \log \hat{p}_i$ | Logistic regression, neural networks |
| KL divergence | $\sum p_i \log(p_i / q_i)$ | Measuring model vs data distribution mismatch |
| Mutual information | $I(X;Y) = H(Y) - H(Y|X)$ | Filter-based feature selection |

---
## Key takeaway

> **Entropy quantifies how mixed a set of labels is; a decision tree splits on the feature that reduces entropy the most, and cross-entropy loss trains classifiers by penalizing low confidence in the correct class.**

---
*Next up: Effect Size & Practical Significance — why a statistically significant result can still be meaningless in practice*